# BP1 Gate 3 — Model / Classifier Benchmark & Champion Selection
**Customer360 Navigator Enterprise Suite — Customer Intent Classification**

## Purpose
Implements Master Execution Plan Section 8 Gate 3: benchmarks a candidate model set on BANKING77's real
77-class intent label (Gate 1's `target_definition.primary_target = "category"`), using **identical CV
folds across every candidate**, and selects a champion by mean CV metric. Nothing here is trained or
evaluated on the CFPB extract - Gate 1's policy is explicit that this classifier trains/evaluates on
BANKING77 alone (CFPB has no narrative text - `RAW_DATA_MANIFEST.md` Finding 2).

## Why TF-IDF, not spaCy / sentence-embeddings / a small transformer
Section 17.5 names "spaCy pipeline, sentence-embedding model, small transformer / classical text
classifier" for the BP1/BP2/BP6 NLP cluster. `00_hardware_benchmark.ipynb`'s live library scan (and this
notebook's own re-check in Section 3) confirms `spacy`, `sentence_transformers`, and `transformers` are
**NOT INSTALLED** on this machine. Per the zero-fabrication/execution-boundary rules, this notebook only
imports what is actually installed - it uses TF-IDF (scikit-learn, confirmed installed) as the real,
available "classical text classifier" feature representation, and flags the missing packages explicitly
rather than silently pretending they were used.

## Why HistGradientBoostingClassifier, not classic GradientBoostingClassifier
Section 17.5's named 6-model set includes "Gradient Boosting." Classic `sklearn.GradientBoostingClassifier`
builds one tree **per class per boosting round** for multiclass problems - with 77 classes that is 77
trees/round, which does not scale to a laptop-safe wall-clock (Section 2's "(New) Execute the entire
pipeline at ultra-fast, laptop-safe wall-clock time" objective). This notebook substitutes
`HistGradientBoostingClassifier` - scikit-learn's own modern, histogram-based replacement, recommended by
scikit-learn itself for problems at this scale - and states this substitution explicitly rather than
silently deviating from the named set.

## Standing rules this notebook follows
- **Execution boundary / zero-fabrication**: Claude wrote this notebook; you run it. Every score below is
  computed live during your run.
- **Identical CV folds** (Gate 3 exit criterion): one `StratifiedKFold` (`configs/resource_limits.yaml`'s
  `cv.n_splits=5`, `random_state=42`) is reused unmodified across all 6 candidates.
- **No nested parallelism** (Section 17.1/17.6 - "exactly one layer of the pipeline is ever parallel at a
  time"): every model's own internal thread count is fixed at 1; the only parallel layer is the outer CV
  fold dispatch, at `configs/hardware_benchmark_summary.json`'s live-recommended `n_jobs`.
- **WARP thread-safety** (`LESSONS_LEARNED_APPLIED.md` #12): the outer CV parallelism uses joblib's
  `threading` backend, not the default process-based `loky` backend, to avoid the same Windows/Jupyter
  process-spawn hang risk already fixed in `00_hardware_benchmark.ipynb`.
- **Leakage discipline** (Gate 1 policy): the TF-IDF vectorizer is fit fresh inside every CV fold's
  training portion only (via an sklearn `Pipeline`, never fit once globally) - fitting it on the full
  train split before CV would leak validation-fold vocabulary/IDF statistics into training. The held-out
  BANKING77 **test** split is touched exactly once, at the very end, only by the already-selected champion.
- **Continue gracefully on failure** (Section 17.7): each candidate's benchmark runs inside its own
  try/except - one model failing (e.g. an unsupported sparse-input path) is recorded and skipped, never
  halts the notebook.
- **Dependency risk avoided**: this project's own live library scan (`configs/hardware_benchmark_summary.json`)
  found several `requirements.txt`-listed packages missing on this machine (duckdb, spacy, transformers,
  numba, shap). BANKING77 (~13K rows, no WARP lazy-scan benefit at this size) is loaded with plain
  `pandas.read_csv` rather than `polars.read_csv().to_pandas()`, which needs `pyarrow` - an avoidable
  dependency risk caught by this notebook's own synthetic-fixture dry-run before delivery.
- **Idempotent**: re-running overwrites this notebook's artifacts and appends/replaces a `gate3_...` block
  in `configs/bp1_customer_intent_classification.yaml` in place, without touching Gate 1's own fields.

## Outputs (idempotent overwrite-in-place)
- `notebooks/bp1_customer_intent_classification/artifacts/gate3_cv_benchmark_results.csv`
- `notebooks/bp1_customer_intent_classification/artifacts/gate3_champion_test_classification_report.json`
- `notebooks/bp1_customer_intent_classification/artifacts/gate3_champion_test_confusion_matrix.csv`
- `notebooks/bp1_customer_intent_classification/artifacts/model_inventory_entry.json` (Gate 3's compliance
  touchpoint - "Model inventory entry opened, SR 11-7 first-line record")
- `configs/bp1_customer_intent_classification.yaml` - `gate3_model_benchmark` block appended/updated

## Prerequisites
BP1 Gate 1 (`..._g1_business_understanding.ipynb`) must have been real-run at least once - this notebook
reads its `target_definition` from `configs/bp1_customer_intent_classification.yaml` and raises if that
field is still `null`. `00_hardware_benchmark.ipynb` must have been real-run for its recommended `n_jobs`.

## If a structural check below fails
It raises `AssertionError` naming the failing check. If the identical-CV-folds check ever fails, that
means a candidate model saw different data than the others and its score is not comparable - do not
silence this check.


In [ ]:
\
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp1_customer_intent_classification/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp1_customer_intent_classification" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import assert_within_ram_ceiling, configure_performance, load_resource_limits  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override (LESSONS_LEARNED_APPLIED.md #12)
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import importlib.util  # noqa: E402
import json  # noqa: E402
import time  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import joblib  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import yaml  # noqa: E402
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier  # noqa: E402
from sklearn.feature_extraction.text import TfidfVectorizer  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import classification_report, confusion_matrix  # noqa: E402
from sklearn.model_selection import StratifiedKFold, cross_validate  # noqa: E402
from sklearn.pipeline import Pipeline  # noqa: E402
from sklearn.preprocessing import FunctionTransformer, LabelEncoder  # noqa: E402
from xgboost import XGBClassifier  # noqa: E402
from lightgbm import LGBMClassifier  # noqa: E402
from catboost import CatBoostClassifier  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

MISSING_NLP_LIBS = [
    name for name in ("spacy", "sentence_transformers", "transformers")
    if importlib.util.find_spec(name) is None
]
print(f"[INFO] NLP libraries NOT installed (confirmed live, not from memory): {MISSING_NLP_LIBS or 'none missing'}")
print("[INFO] Using TF-IDF (scikit-learn) as the classical text-classifier feature representation.")

# ============================================================
# SECTION 4: Load Gate 1's policy + the hardware benchmark's live recommendation - never hardcode
# ============================================================
bp1_config_path = CONFIGS_DIR / "bp1_customer_intent_classification.yaml"
with open(bp1_config_path, "r", encoding="utf-8") as f:
    bp1_config = yaml.safe_load(f)
target_def = bp1_config.get("target_definition")
assert target_def is not None, (
    "[CHECK FAILED] target_definition is still null in configs/bp1_customer_intent_classification.yaml - "
    "run BP1 Gate 1 (..._g1_business_understanding.ipynb) first."
)
PRIMARY_TARGET = target_def["primary_target"]
FEATURE_COL = target_def["feature_variable"]
print(f"[OK] Gate 1 policy loaded: target='{PRIMARY_TARGET}', feature='{FEATURE_COL}'")

hw_summary_path = CONFIGS_DIR / "hardware_benchmark_summary.json"
with open(hw_summary_path, "r", encoding="utf-8") as f:
    hw_summary = json.load(f)
N_JOBS = hw_summary["recommended_configuration"]["recommended_n_jobs"]
print(f"[OK] Hardware benchmark recommendation loaded: n_jobs={N_JOBS} "
      f"(never hardcoded - see {hw_summary_path.relative_to(PROJECT_ROOT)})")

cv_settings = RESOURCE_LIMITS["cv"]
print(f"[OK] CV settings loaded: n_splits={cv_settings['n_splits']}, "
      f"random_state={cv_settings['random_state']}")

# ============================================================
# SECTION 5: Load BANKING77 train/test (real split, never re-randomized - Gate 1 policy)
# ============================================================
B77_TRAIN_PATH = DATA_EXTERNAL_DIR / "banking77_train.csv"
B77_TEST_PATH = DATA_EXTERNAL_DIR / "banking77_test.csv"


# Plain pandas here (not polars.read_csv().to_pandas()) - this project's OWN live library scan found
# several requirements.txt-listed packages missing on this machine (duckdb, spacy, transformers, numba,
# shap all showed "NOT INSTALLED" in configs/hardware_benchmark_summary.json); polars' to_pandas()
# requires pyarrow, an avoidable dependency risk for this small (~13K-row) file where WARP's lazy-scan
# benefit does not apply anyway. Caught by this notebook's own synthetic-fixture dry-run before delivery.
train_df = pd.read_csv(B77_TRAIN_PATH, dtype={"text": str, "category": str})
test_df = pd.read_csv(B77_TEST_PATH, dtype={"text": str, "category": str})

overlap = set(train_df[FEATURE_COL]) & set(test_df[FEATURE_COL])
assert len(overlap) == 0, f"[CHECK FAILED] {len(overlap)} exact-text rows overlap train/test - leakage risk."
print(f"[OK] Re-verified zero train/test text overlap (train={len(train_df):,}, test={len(test_df):,}).")

# Real bug caught by this notebook's own synthetic-fixture dry-run before delivery: XGBoost's sklearn API
# (this environment's real installed version) requires y as contiguous integers 0..n_classes-1 - it raises
# "Invalid classes inferred from unique values of y" on raw string labels, unlike LogisticRegression/
# RandomForest/LightGBM/CatBoost, which all handle string labels natively. A single shared LabelEncoder is
# used for every candidate (not just XGBoost) so every model sees identical target encoding; original
# string category names are restored via inverse_transform for all human-readable output below.
unseen_test_categories = set(test_df[PRIMARY_TARGET]) - set(train_df[PRIMARY_TARGET])
assert not unseen_test_categories, (
    f"[CHECK FAILED] Test split has categories never seen in train: {unseen_test_categories} - "
    "cannot encode/predict these."
)
label_encoder = LabelEncoder().fit(train_df[PRIMARY_TARGET])
X_train, y_train = train_df[FEATURE_COL], label_encoder.transform(train_df[PRIMARY_TARGET])
X_test, y_test_labels = test_df[FEATURE_COL], test_df[PRIMARY_TARGET]
y_test = label_encoder.transform(y_test_labels)
print(f"[OK] Labels integer-encoded ({len(label_encoder.classes_)} classes) - same encoding used for every "
      "candidate model; original category names restored for all reporting output.")

# ============================================================
# SECTION 6: Candidate model set (6 models, mirroring Section 17.5's named set) - internal n_jobs=1
# on every candidate (no nested parallelism); the only parallel layer is the outer CV dispatch below.
# ============================================================
TFIDF_KWARGS = dict(max_features=5000, ngram_range=(1, 2), min_df=2, sublinear_tf=True, stop_words="english")

CANDIDATES = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=cv_settings["random_state"]),
    "random_forest": RandomForestClassifier(
        n_estimators=100, max_depth=20, n_jobs=1, random_state=cv_settings["random_state"]
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=100, random_state=cv_settings["random_state"]
    ),
    "xgboost": XGBClassifier(
        n_estimators=100, max_depth=6, n_jobs=1, verbosity=0, random_state=cv_settings["random_state"]
    ),
    "lightgbm": LGBMClassifier(
        n_estimators=100, n_jobs=1, verbose=-1, random_state=cv_settings["random_state"]
    ),
    "catboost": CatBoostClassifier(
        iterations=100, thread_count=1, verbose=False, allow_writing_files=False,
        random_state=cv_settings["random_state"],
    ),
}

# Real bug caught by this notebook's own synthetic-fixture dry-run before delivery: this environment's
# real installed scikit-learn version raises "Sparse data was passed for X, but dense data is required"
# for HistGradientBoostingClassifier - unlike every other candidate here, it does not accept the sparse
# TF-IDF matrix directly. A densify step is added to ONLY that one candidate's pipeline (max_features=5000
# keeps a dense fold-sized matrix to a bounded, transient ~ a few hundred MB - well within the WARP RAM
# ceiling - and it is never persisted, only used during that model's own fit/predict calls).
NEEDS_DENSE = {"hist_gradient_boosting"}


def _to_dense(x):
    return x.toarray() if hasattr(x, "toarray") else x

# ============================================================
# SECTION 7: Identical CV folds across every candidate (Gate 3 exit criterion)
# ============================================================
skf = StratifiedKFold(
    n_splits=cv_settings["n_splits"], shuffle=cv_settings["shuffle"], random_state=cv_settings["random_state"]
)
SCORING = ["f1_macro", "f1_weighted", "accuracy"]

cv_results_rows = []
failed_candidates = []
for name, model in CANDIDATES.items():
    pipeline_steps = [("tfidf", TfidfVectorizer(**TFIDF_KWARGS))]
    if name in NEEDS_DENSE:
        pipeline_steps.append(("densify", FunctionTransformer(_to_dense, accept_sparse=True)))
    pipeline_steps.append(("clf", model))
    pipeline = Pipeline(pipeline_steps)
    print(f"\n[BENCH] starting {name} ({cv_settings['n_splits']}-fold CV, n_jobs={N_JOBS}, threading backend)...")
    t0 = time.perf_counter()
    try:
        with joblib.parallel_backend("threading", n_jobs=N_JOBS):
            scores = cross_validate(pipeline, X_train, y_train, cv=skf, scoring=SCORING, n_jobs=N_JOBS)
        elapsed = time.perf_counter() - t0
        row = {
            "model": name,
            "status": "OK",
            "elapsed_seconds": round(elapsed, 2),
            "mean_f1_macro": round(float(np.mean(scores["test_f1_macro"])), 4),
            "std_f1_macro": round(float(np.std(scores["test_f1_macro"])), 4),
            "mean_f1_weighted": round(float(np.mean(scores["test_f1_weighted"])), 4),
            "mean_accuracy": round(float(np.mean(scores["test_accuracy"])), 4),
        }
        print(f"[BENCH] {name} done in {elapsed:.1f}s: mean f1_macro={row['mean_f1_macro']} "
              f"(+/- {row['std_f1_macro']})")
    except Exception as e:  # noqa: BLE001 - continue gracefully on an individual model failure (Section 17.7)
        elapsed = time.perf_counter() - t0
        row = {
            "model": name, "status": f"FAILED: {type(e).__name__}: {e}", "elapsed_seconds": round(elapsed, 2),
            "mean_f1_macro": None, "std_f1_macro": None, "mean_f1_weighted": None, "mean_accuracy": None,
        }
        failed_candidates.append(name)
        print(f"[FAILED] {name} after {elapsed:.1f}s: {type(e).__name__}: {e} - continuing with remaining models.")
    cv_results_rows.append(row)

cv_results_df = pd.DataFrame(cv_results_rows)
print("\n=== CV BENCHMARK RESULTS (identical folds across all candidates) ===")
print(cv_results_df.to_string(index=False))

results_csv_path = ARTIFACTS_DIR / "gate3_cv_benchmark_results.csv"
cv_results_df.to_csv(results_csv_path, index=False)
print(f"\n[SAVED] {results_csv_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 8: Champion selection - highest mean CV f1_macro (class imbalance -> macro F1, not accuracy)
# ============================================================
passing_results = cv_results_df[cv_results_df["status"] == "OK"]
assert len(passing_results) > 0, "[CHECK FAILED] Every candidate model failed - nothing to select a champion from."
champion_row = passing_results.loc[passing_results["mean_f1_macro"].idxmax()]
champion_name = champion_row["model"]
print(f"\n[RESULT] Champion: {champion_name} (mean CV f1_macro={champion_row['mean_f1_macro']})")

# ============================================================
# SECTION 9: Refit champion on the FULL train split, evaluate ONCE on the held-out real test split
# ============================================================
assert_within_ram_ceiling(RESOURCE_LIMITS)
champion_steps = [("tfidf", TfidfVectorizer(**TFIDF_KWARGS))]
if champion_name in NEEDS_DENSE:
    champion_steps.append(("densify", FunctionTransformer(_to_dense, accept_sparse=True)))
champion_steps.append(("clf", CANDIDATES[champion_name]))
champion_pipeline = Pipeline(champion_steps)
print(f"\n[FINAL] Refitting champion ({champion_name}) on the full train split ({len(X_train):,} rows)...")
t0 = time.perf_counter()
champion_pipeline.fit(X_train, y_train)
fit_elapsed = time.perf_counter() - t0
print(f"[FINAL] Fit done in {fit_elapsed:.1f}s. Evaluating ONCE on the held-out test split ({len(X_test):,} rows)...")

y_pred_encoded = champion_pipeline.predict(X_test)
# Decode back to real category names for every human-readable output below - y_test/y_pred were integer-
# encoded only because XGBoost's sklearn API requires it (Section 5); reporting should never show class
# indices when the real label strings are available.
y_pred = label_encoder.inverse_transform(y_pred_encoded)
test_report = classification_report(y_test_labels, y_pred, output_dict=True, zero_division=0)
test_f1_macro = test_report["macro avg"]["f1-score"]
test_f1_weighted = test_report["weighted avg"]["f1-score"]
test_accuracy = test_report["accuracy"]
print(f"[FINAL] Held-out test: f1_macro={test_f1_macro:.4f}, f1_weighted={test_f1_weighted:.4f}, "
      f"accuracy={test_accuracy:.4f}")

labels_sorted = sorted(set(y_test_labels) | set(y_pred))
cm = confusion_matrix(y_test_labels, y_pred, labels=labels_sorted)
cm_df = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)

confused_pairs = []
for i, true_label in enumerate(labels_sorted):
    for j, pred_label in enumerate(labels_sorted):
        if i != j and cm[i, j] > 0:
            confused_pairs.append((true_label, pred_label, int(cm[i, j])))
confused_pairs.sort(key=lambda t: t[2], reverse=True)
top_confused = confused_pairs[:10]
print("\n[FINAL] Top confused (true -> predicted) pairs on the held-out test split:")
for true_label, pred_label, count in top_confused:
    print(f"  {true_label} -> {pred_label}: {count}")

# ============================================================
# SECTION 10: Write outputs (idempotent overwrite-in-place)
# ============================================================
report_json_path = ARTIFACTS_DIR / "gate3_champion_test_classification_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(test_report, f, indent=2)
print(f"\n[SAVED] {report_json_path.relative_to(PROJECT_ROOT)}")

cm_csv_path = ARTIFACTS_DIR / "gate3_champion_test_confusion_matrix.csv"
cm_df.to_csv(cm_csv_path)
print(f"[SAVED] {cm_csv_path.relative_to(PROJECT_ROOT)}")

model_inventory_entry = {
    "bp_id": "bp1",
    "gate": 3,
    "compliance_touchpoint": "Model inventory entry opened (SR 11-7 first-line record)",
    "model_name": champion_name,
    "model_family": "TF-IDF + " + champion_name,
    "target_variable": PRIMARY_TARGET,
    "feature_variable": FEATURE_COL,
    "training_data": "PolyAI BANKING77 train split (real, provided split, never re-randomized)",
    "n_train_rows": len(X_train),
    "n_test_rows": len(X_test),
    "n_classes": len(label_encoder.classes_),
    "cv_folds": cv_settings["n_splits"],
    "cv_mean_f1_macro": float(champion_row["mean_f1_macro"]),
    "held_out_test_f1_macro": float(test_f1_macro),
    "held_out_test_f1_weighted": float(test_f1_weighted),
    "held_out_test_accuracy": float(test_accuracy),
    "candidates_evaluated": list(CANDIDATES.keys()),
    "candidates_failed": failed_candidates,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "first-line record only - Gate 4 independent-style statistical validation not yet performed",
}
inventory_path = ARTIFACTS_DIR / "model_inventory_entry.json"
with open(inventory_path, "w", encoding="utf-8") as f:
    json.dump(model_inventory_entry, f, indent=2)
print(f"[SAVED] {inventory_path.relative_to(PROJECT_ROOT)}")

# Order-independent patch of bp1_customer_intent_classification.yaml - replaces ONLY this
# gate's own marker-delimited block, preserving the front matter and every other gate's block
# regardless of position (src/utils/bp1_config_sync.py; see LESSONS_LEARNED_APPLIED.md #20 for
# the real incident the previous split()-based approach here was vulnerable to).
import re  # noqa: E402
from utils.bp1_config_sync import write_gate_block  # noqa: E402

new_status_value = "gate1_confirmed_gate2_confirmed_gate3_confirmed"
config_text = bp1_config_path.read_text(encoding="utf-8")
config_text = re.sub(r'^status:.*$', f'status: "{new_status_value}"', config_text, count=1, flags=re.MULTILINE)
bp1_config_path.write_text(config_text, encoding="utf-8")

gate3_marker = "# --- Gate 3 (Model/Classifier Benchmark) results (appended, idempotent overwrite) ---"
gate3_block_lines = [
    "gate3_model_benchmark:",
    f'  champion_model: "{champion_name}"',
    f"  cv_mean_f1_macro: {champion_row['mean_f1_macro']}",
    f"  held_out_test_f1_macro: {round(test_f1_macro, 4)}",
    f"  held_out_test_f1_weighted: {round(test_f1_weighted, 4)}",
    f"  held_out_test_accuracy: {round(test_accuracy, 4)}",
    f"  candidates_evaluated: {list(CANDIDATES.keys())}",
    f"  candidates_failed: {failed_candidates}",
    f'  generated_at_utc: "{datetime.now(timezone.utc).isoformat()}"',
]
write_gate_block(bp1_config_path, gate3_marker, gate3_block_lines)
print(f"[SAVED] {bp1_config_path.relative_to(PROJECT_ROOT)} (gate3_model_benchmark block)")

# ============================================================
# SECTION 11: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "at_least_one_candidate_passed": len(passing_results) > 0,
    "identical_cv_object_used_for_all_candidates": True,  # by construction - same `skf` instance, Section 7
    "champion_selected_by_mean_cv_f1_macro": champion_name in CANDIDATES,
    "held_out_test_touched_exactly_once": True,  # by construction - Section 9 is the only X_test/y_test use
    "cv_results_csv_written": results_csv_path.exists(),
    "classification_report_json_written": report_json_path.exists(),
    "confusion_matrix_csv_written": cm_csv_path.exists(),
    "model_inventory_entry_written": inventory_path.exists(),
    "bp1_config_yaml_updated": bp1_config_path.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(f"\n[ALL CHECKS PASSED] BP1 Gate 3 complete. Champion: {champion_name} "
      f"(held-out test f1_macro={round(test_f1_macro, 4)}). "
      f"{len(failed_candidates)} candidate(s) failed and were skipped: {failed_candidates or 'none'}. "
      "Proceed to BP1 Gate 4 (Statistical Validation & Explainability) next.")
